# Financial indicators: several inputs and outputs

Many indicators do not fit the one-series-in, one-series-out shape. Some need
several price series at once: ATR and the Stochastic oscillator read high, low,
and close together. Some return several components at once: Bollinger Bands give a
lower, middle, and upper band, and MACD gives a line, a signal, and a histogram.

screamer handles both with one calling convention. Several inputs go in as N
separate arrays or as a single `(T, N)` matrix. Several outputs come back as a
`(T, M)` array whose trailing axis indexes the components, which you unpack column
by column. This notebook shows both patterns on real indicators; the full catalog
is in the [function reference](../by_topic_index.rst).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from screamer import ATR, MACD, BollingerBands

rng = np.random.default_rng(2)
close = 100 + np.cumsum(rng.standard_normal(400))
high  = close + np.abs(rng.standard_normal(400))
low   = close - np.abs(rng.standard_normal(400))

## Several outputs: components from one series

`BollingerBands` and `MACD` each take a single price series and return a
`(T, 3)` array. The trailing axis holds the components, so transposing lets you
unpack them into named series in one line.

In [ ]:
bb = BollingerBands(20)(close)
lower, mid, upper = bb.T                 # unpack the three columns

macd = MACD()(close)
macd_line, macd_signal, macd_hist = macd.T

print(f"BollingerBands -> {bb.shape}: lower, mid, upper")
print(f"MACD           -> {macd.shape}: line, signal, histogram")

## Several inputs: OHLC as arrays or a matrix

`ATR` needs high, low, and close. Pass them as three separate arrays, or stack
them into a single `(T, 3)` matrix and pass that. The two forms are equivalent.

In [ ]:
atr = ATR(14)(high, low, close)              # three separate series
hlc = np.column_stack([high, low, close])    # or one (T, 3) matrix
np.testing.assert_array_equal(atr, ATR(14)(hlc))

print(f"both input forms agree; ATR -> {atr.shape}")

## Seeing the patterns together

The top panel unpacks a multi-output indicator (the three Bollinger components),
the middle panel shows another (MACD's line, signal, and histogram), and the
bottom panel is a multi-input indicator (ATR, computed from high, low, close).

In [ ]:
xs = np.arange(len(close))

fig, ax = plt.subplots(3, 1, figsize=(9, 7), sharex=True)

ax[0].plot(close, lw=0.7, color="0.4", label="close")
ax[0].plot(mid, color="crimson", lw=0.4, alpha=0.7)
ax[0].plot(lower, "r--", lw=0.5)
ax[0].plot(upper, "r--", lw=0.5)
ax[0].fill_between(xs, lower, upper, alpha=0.1, color="r")
ax[0].set_title("BollingerBands(20): one series in, three components out")
ax[0].legend(loc="upper left", fontsize=8)

ax[1].plot(macd_line, lw=0.8, label="macd")
ax[1].plot(macd_signal, lw=0.8, label="signal")
ax[1].bar(xs, macd_hist, color="0.7", width=1.0, label="histogram")
ax[1].axhline(0, color="k", lw=0.4)
ax[1].set_title("MACD(): line, signal, histogram")
ax[1].legend(loc="upper left", fontsize=8)

ax[2].plot(atr, lw=0.8, label="ATR(14)")
ax[2].set_title("ATR(14): three series in (high, low, close)")
ax[2].set_xlabel("sample")
ax[2].legend(loc="upper left", fontsize=8)

plt.tight_layout()